In [1]:
import pandas as pd

path = "../../data/Data_Entry_2017.csv"

df = pd.read_csv(path)

df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [2]:
print("Number of X-ray images:", len(df))

Number of X-ray images: 112120


In [3]:
print("Number of unique patients:", df["Patient ID"].nunique())

Number of unique patients: 30805


In [4]:
labels = df["Finding Labels"].str.split("|").explode()

print("Number of unique labels:", labels.nunique())
print("\nLabels:")
print(sorted(labels.unique()))

Number of unique labels: 15

Labels:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


In [5]:
label_counts = labels.value_counts()

print(label_counts)

Finding Labels
No Finding            60361
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
Name: count, dtype: int64


In [6]:
selected_classes = [
    "Atelectasis",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule"
]

def contains_selected(label):
    labels = label.split("|")
    return [x for x in labels if x in selected_classes]

df["selected_labels"] = df["Finding Labels"].apply(contains_selected)

selected_df = df[df["selected_labels"].str.len() > 0].copy()

print("Images containing at least one selected disease:", len(selected_df))
print(
    "Images containing 2+ selected diseases:",
    (selected_df["selected_labels"].str.len() >= 2).sum()
)

Images containing at least one selected disease: 42205
Images containing 2+ selected diseases: 12021


In [7]:
print("Unique patients in selected data:",
      selected_df["Patient ID"].nunique())

Unique patients in selected data: 12668


In [8]:
patient_counts = selected_df["Patient ID"].value_counts()

print(patient_counts.describe())

count    12668.000000
mean         3.331623
std          5.734486
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        106.000000
Name: count, dtype: float64


In [9]:
no_finding_count = (df["Finding Labels"] == "No Finding").sum()

print("No Finding images:", no_finding_count)

No Finding images: 60361


In [10]:
selected_mask = df["selected_labels"].str.len() > 0

other_disease_df = df[
    (~selected_mask) &
    (df["Finding Labels"] != "No Finding")
].copy()

print("Images with other diseases only:", len(other_disease_df))

Images with other diseases only: 9554


In [11]:
print(
    "Unique patients with No Finding:",
    df.loc[df["Finding Labels"] == "No Finding", "Patient ID"].nunique()
)

Unique patients with No Finding: 24907


In [12]:
selected_mask = df["selected_labels"].str.len() > 0

other_disease_df = df[
    (~selected_mask) &
    (df["Finding Labels"] != "No Finding")
].copy()

print("Images with other diseases only:", len(other_disease_df))

Images with other diseases only: 9554


In [13]:
target_patients = set(selected_df["Patient ID"])

no_finding_patients = set(
    df.loc[df["Finding Labels"] == "No Finding", "Patient ID"]
)

overlap = target_patients & no_finding_patients

print("Patients appearing in both groups:", len(overlap))

Patients appearing in both groups: 7779


In [14]:
selected_classes = [
    "Atelectasis",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule"
]

for disease in selected_classes:
    df[disease] = df["Finding Labels"].apply(
        lambda x: int(disease in x.split("|"))
    )

df[
    ["Image Index", "Patient ID", "Finding Labels"] + selected_classes
].head(10)

,Image Index,Patient ID,Finding Labels,Atelectasis,Effusion,Infiltration,Mass,Nodule
0,00000001_000.png,1,Cardiomegaly,0,0,0,0,0
1,00000001_001.png,1,Cardiomegaly|Emphysema,0,0,0,0,0
2,00000001_002.png,1,Cardiomegaly|Effusion,0,1,0,0,0
3,00000002_000.png,2,No Finding,0,0,0,0,0
4,00000003_000.png,3,Hernia,0,0,0,0,0
5,00000003_001.png,3,Hernia,0,0,0,0,0
6,00000003_002.png,3,Hernia,0,0,0,0,0
7,00000003_003.png,3,Hernia|Infiltration,0,0,1,0,0
8,00000003_004.png,3,Hernia,0,0,0,0,0
9,00000003_005.png,3,Hernia,0,0,0,0,0


In [15]:
print("Total rows:", len(df))

Total rows: 112120


In [16]:
print("Target label counts:")
print(df[selected_classes].sum())

Target label counts:
Atelectasis     11559
Effusion        13317
Infiltration    19894
Mass             5782
Nodule           6331
dtype: int64


In [17]:
target_mask = df[selected_classes].sum(axis=1) > 0
no_finding_mask = df["Finding Labels"] == "No Finding"

final_df = df[target_mask | no_finding_mask].copy()

print("Final dataset images:", len(final_df))
print("Final dataset patients:", final_df["Patient ID"].nunique())

Final dataset images: 102566
Final dataset patients: 29796


In [18]:
from sklearn.model_selection import train_test_split

patient_ids = final_df["Patient ID"].unique()

train_patients, temp_patients = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

Train patients: 20857
Validation patients: 4469
Test patients: 4470


In [19]:
train_df = final_df[final_df["Patient ID"].isin(train_patients)].copy()
val_df = final_df[final_df["Patient ID"].isin(val_patients)].copy()
test_df = final_df[final_df["Patient ID"].isin(test_patients)].copy()

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Train images: 71766
Validation images: 15204
Test images: 15596


In [20]:
print(
    "Train ∩ Validation:",
    len(set(train_df["Patient ID"]) & set(val_df["Patient ID"]))
)

print(
    "Train ∩ Test:",
    len(set(train_df["Patient ID"]) & set(test_df["Patient ID"]))
)

print(
    "Validation ∩ Test:",
    len(set(val_df["Patient ID"]) & set(test_df["Patient ID"]))
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [21]:
import os

selected_classes = [
    "Atelectasis",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule"
]

# ---------- 1. Verify patient separation ----------
train_patient_set = set(train_df["Patient ID"])
val_patient_set = set(val_df["Patient ID"])
test_patient_set = set(test_df["Patient ID"])

print("Patient overlap checks:")
print("Train ∩ Validation:", len(train_patient_set & val_patient_set))
print("Train ∩ Test:", len(train_patient_set & test_patient_set))
print("Validation ∩ Test:", len(val_patient_set & test_patient_set))

# ---------- 2. Dataset sizes ----------
print("\nDataset sizes:")

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        f"{name}: {len(data):,} images | "
        f"{data['Patient ID'].nunique():,} patients"
    )

# ---------- 3. Label distribution ----------
print("\nTarget-label distribution:")

for disease in selected_classes:
    print(
        f"{disease:15s} "
        f"Train: {train_df[disease].sum():6,} | "
        f"Val: {val_df[disease].sum():6,} | "
        f"Test: {test_df[disease].sum():6,}"
    )

# ---------- 4. No Finding distribution ----------
print("\nNo Finding:")
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    count = (data["Finding Labels"] == "No Finding").sum()
    print(f"{name}: {count:,}")

# ---------- 5. Save splits ----------
split_dir = "../../data/splits"
os.makedirs(split_dir, exist_ok=True)

train_df.to_csv(f"{split_dir}/train.csv", index=False)
val_df.to_csv(f"{split_dir}/validation.csv", index=False)
test_df.to_csv(f"{split_dir}/test.csv", index=False)

print("\nSaved:")
print("train.csv")
print("validation.csv")
print("test.csv")

Patient overlap checks:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Dataset sizes:
Train: 71,766 images | 20,857 patients
Validation: 15,204 images | 4,469 patients
Test: 15,596 images | 4,470 patients

Target-label distribution:
Atelectasis     Train:  8,033 | Val:  1,729 | Test:  1,797
Effusion        Train:  9,193 | Val:  1,976 | Test:  2,148
Infiltration    Train: 13,928 | Val:  2,934 | Test:  3,032
Mass            Train:  3,980 | Val:    805 | Test:    997
Nodule          Train:  4,580 | Val:    842 | Test:    909

No Finding:
Train: 42,325
Validation: 9,020
Test: 9,016

Saved:
train.csv
validation.csv
test.csv


In [22]:
import os
from pathlib import Path
from collections import defaultdict

# --------------------------------------------------
# 1. Find which image folders actually exist
# --------------------------------------------------
data_dir = Path("../../data")

image_dirs = [
    data_dir / f"images_{i:03d}"
    for i in range(1, 13)
]

existing_dirs = [d for d in image_dirs if d.exists()]

print("Image folders found:", len(existing_dirs))
for d in existing_dirs:
    print(" ", d)

# --------------------------------------------------
# 2. Count PNG files in each folder
# --------------------------------------------------
image_index = {}

for folder in existing_dirs:
    for img_path in folder.glob("*.png"):
        image_index[img_path.name] = str(img_path)

print("\nTotal indexed images:", len(image_index))

# --------------------------------------------------
# 3. Check whether our split CSVs have image files
# --------------------------------------------------
for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    found = split_df["Image Index"].isin(image_index).sum()
    missing = len(split_df) - found

    print(
        f"\n{name}: "
        f"{found:,} found | "
        f"{missing:,} missing"
    )

# --------------------------------------------------
# 4. Show a few examples
# --------------------------------------------------
print("\nSample image paths:")

for img_name in train_df["Image Index"].head(5):
    print(img_name, "->", image_index.get(img_name))

Image folders found: 0

Total indexed images: 0

Train: 0 found | 71,766 missing

Validation: 0 found | 15,204 missing

Test: 0 found | 15,596 missing

Sample image paths:
00000001_002.png -> None
00000002_000.png -> None
00000003_003.png -> None
00000005_000.png -> None
00000005_001.png -> None


In [23]:
from pathlib import Path

# Location of the downloaded NIH ChestX-ray14 dataset
dataset_root = Path(r"D:\archive")

# Build the 12 image-folder paths
image_dirs = [
    dataset_root / f"images_{i:03d}" / "images"
    for i in range(1, 13)
]

# Check that all folders exist
print("Image folders:")
for folder in image_dirs:
    print(f"{folder} -> {folder.exists()}")

# Index every PNG image
image_index = {}

for folder in image_dirs:
    for img_path in folder.glob("*.png"):
        image_index[img_path.name] = img_path

print("\nTotal images indexed:", len(image_index))

# Check our three splits
for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    found = split_df["Image Index"].isin(image_index).sum()
    missing = len(split_df) - found

    print(f"{name}: {found:,} found | {missing:,} missing")

# Show a few examples
print("\nSample paths:")
for img_name in train_df["Image Index"].head(5):
    print(img_name, "->", image_index.get(img_name))

Image folders:
D:\archive\images_001\images -> True
D:\archive\images_002\images -> True
D:\archive\images_003\images -> True
D:\archive\images_004\images -> True
D:\archive\images_005\images -> True
D:\archive\images_006\images -> True
D:\archive\images_007\images -> True
D:\archive\images_008\images -> True
D:\archive\images_009\images -> True
D:\archive\images_010\images -> True
D:\archive\images_011\images -> True
D:\archive\images_012\images -> True

Total images indexed: 112120
Train: 71,766 found | 0 missing
Validation: 15,204 found | 0 missing
Test: 15,596 found | 0 missing

Sample paths:
00000001_002.png -> D:\archive\images_001\images\00000001_002.png
00000002_000.png -> D:\archive\images_001\images\00000002_000.png
00000003_003.png -> D:\archive\images_001\images\00000003_003.png
00000005_000.png -> D:\archive\images_001\images\00000005_000.png
00000005_001.png -> D:\archive\images_001\images\00000005_001.png


In [24]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# --------------------------------------------------
# 1. Image transformations
# --------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------------------------------------
# 2. PyTorch Dataset
# --------------------------------------------------

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_index, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_index = image_index
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_name = row["Image Index"]
        image_path = self.image_index[image_name]

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply preprocessing
        if self.transform:
            image = self.transform(image)

        # Multi-label target
        labels = torch.tensor(
            row[selected_classes].values.astype("float32")
        )

        return image, labels

# --------------------------------------------------
# 3. Create datasets
# --------------------------------------------------

train_dataset = ChestXrayDataset(
    train_df,
    image_index,
    transform=train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_index,
    transform=eval_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_index,
    transform=eval_transform
)

# --------------------------------------------------
# 4. Create DataLoaders
# --------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# --------------------------------------------------
# 5. Test one sample
# --------------------------------------------------

sample_image, sample_labels = train_dataset[0]

print("Single image shape :", sample_image.shape)
print("Single labels       :", sample_labels)
print("Image dtype         :", sample_image.dtype)
print("Label dtype         :", sample_labels.dtype)

# --------------------------------------------------
# 6. Test one batch
# --------------------------------------------------

images, labels = next(iter(train_loader))

print("\nBatch image shape   :", images.shape)
print("Batch label shape   :", labels.shape)

# --------------------------------------------------
# 7. Check GPU
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Single image shape : torch.Size([3, 224, 224])
Single labels       : tensor([0., 1., 0., 0., 0.])
Image dtype         : torch.float32
Label dtype         : torch.float32

Batch image shape   : torch.Size([16, 3, 224, 224])
Batch label shape   : torch.Size([16, 5])

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [25]:
import sys
import torch

print("Python executable:")
print(sys.executable)

print("\nPyTorch version:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

print("\nCUDA version:")
print(torch.version.cuda)

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

Python executable:
c:\Users\Acer\Desktop\Chest-XRay-Project\venv\Scripts\python.exe

PyTorch version:
2.14.0+cu126

CUDA available:
True

CUDA version:
12.6

GPU:
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [26]:
import torch
import torch.nn as nn
from torchvision import models

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained DenseNet121
model = models.densenet121(weights="DEFAULT")

# Number of input features to the original classifier
num_features = model.classifier.in_features

# Replace the original classifier
model.classifier = nn.Linear(
    num_features,
    len(selected_classes)
)

# Move model to GPU
model = model.to(device)

print("Device:", device)
print("Output classes:", selected_classes)
print("Number of outputs:", len(selected_classes))

Device: cuda
Output classes: ['Atelectasis', 'Effusion', 'Infiltration', 'Mass', 'Nodule']
Number of outputs: 5


In [27]:
# Get one batch from our training DataLoader
images, labels = next(iter(train_loader))

# Move the batch to the GPU
images = images.to(device)
labels = labels.to(device)

# Put the model in evaluation mode temporarily
model.eval()

# Disable gradient calculation because we're only testing inference
with torch.no_grad():
    outputs = model(images)

print("Input shape :", images.shape)
print("Label shape :", labels.shape)
print("Output shape:", outputs.shape)
print("Device      :", images.device)

print("\nRaw model outputs:")
print(outputs[0])

Input shape : torch.Size([16, 3, 224, 224])
Label shape : torch.Size([16, 5])
Output shape: torch.Size([16, 5])
Device      : cuda:0

Raw model outputs:
tensor([-0.4832, -0.2389,  0.6043, -0.6363,  1.0507], device='cuda:0')


In [28]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Loss:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

Loss: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.0001


In [29]:
import torch

# Make sure model is in training mode
model.train()

# Get one batch
images, labels = next(iter(train_loader))

# Move data to GPU
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)

# Clear old gradients
optimizer.zero_grad(set_to_none=True)

# Forward pass
outputs = model(images)

# Calculate loss
loss = criterion(outputs, labels)

# Backpropagation
loss.backward()

# Update model weights
optimizer.step()

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
print("Loss        :", loss.item())
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Input shape : torch.Size([16, 3, 224, 224])
Output shape: torch.Size([16, 5])
Loss        : 0.7734166979789734
GPU memory allocated: 0.13 GB


In [30]:
for disease in selected_classes:
    positives = train_df[disease].sum()
    negatives = len(train_df) - positives

    print(
        f"{disease:15s} | "
        f"Positive: {positives:6,} | "
        f"Negative: {negatives:6,} | "
        f"Positive %: {positives / len(train_df) * 100:.2f}%"
    )

Atelectasis     | Positive:  8,033 | Negative: 63,733 | Positive %: 11.19%
Effusion        | Positive:  9,193 | Negative: 62,573 | Positive %: 12.81%
Infiltration    | Positive: 13,928 | Negative: 57,838 | Positive %: 19.41%
Mass            | Positive:  3,980 | Negative: 67,786 | Positive %: 5.55%
Nodule          | Positive:  4,580 | Negative: 67,186 | Positive %: 6.38%


In [31]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# --------------------------------------------------
# 1. Calculate class weights from TRAINING data
# --------------------------------------------------

positive_counts = train_df[selected_classes].sum().values
negative_counts = len(train_df) - positive_counts

pos_weight = torch.tensor(
    negative_counts / positive_counts,
    dtype=torch.float32,
    device=device
)

print("Positive weights:")
for disease, weight in zip(selected_classes, pos_weight):
    print(f"{disease:15s}: {weight.item():.2f}")

# --------------------------------------------------
# 2. Weighted BCE loss
# --------------------------------------------------

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# --------------------------------------------------
# 3. Optimizer
# --------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# --------------------------------------------------
# 4. Mixed precision scaler
# --------------------------------------------------

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

# --------------------------------------------------
# 5. Training configuration
# --------------------------------------------------

num_epochs = 10
early_stopping_patience = 3

print("\nTraining configuration:")
print("Epochs:", num_epochs)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("AMP enabled:", torch.cuda.is_available())

Positive weights:
Atelectasis    : 7.93
Effusion       : 6.81
Infiltration   : 4.15
Mass           : 17.03
Nodule         : 14.67

Training configuration:
Epochs: 10
Learning rate: 0.0001
Weight decay: 1e-05
AMP enabled: True


In [32]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ============================================================
# 1. Reset DenseNet121 to clean pretrained weights
# ============================================================

model = models.densenet121(weights="DEFAULT")

num_features = model.classifier.in_features

model.classifier = nn.Linear(
    num_features,
    len(selected_classes)
)

model = model.to(device)

print("Model reset.")
print("Device:", device)
print("Classes:", selected_classes)


# ============================================================
# 2. Calculate class weights
# ============================================================

positive_counts = train_df[selected_classes].sum().values
negative_counts = len(train_df) - positive_counts

pos_weight = torch.tensor(
    negative_counts / positive_counts,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)


# ============================================================
# 3. Optimizer
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)


# ============================================================
# 4. Mixed Precision
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


# ============================================================
# 5. Enable cuDNN benchmark
# ============================================================

torch.backends.cudnn.benchmark = True


# ============================================================
# 6. Smoke-test training
# ============================================================

model.train()

smoke_test_batches = 100
running_loss = 0.0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader):

    # Stop after 100 batches
    if batch_idx >= smoke_test_batches:
        break

    # Move data to GPU
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    # Clear gradients
    optimizer.zero_grad(set_to_none=True)

    # Mixed-precision forward pass
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        outputs = model(images)
        loss = criterion(outputs, labels)

    # Backpropagation
    scaler.scale(loss).backward()

    # Update weights
    scaler.step(optimizer)
    scaler.update()

    running_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Batch {batch_idx + 1:3d}/{smoke_test_batches} "
            f"| Loss: {loss.item():.4f}"
        )

elapsed = time.time() - start_time

print("\n" + "=" * 50)
print("SMOKE TEST COMPLETE")
print("=" * 50)

print(f"Average loss: {running_loss / smoke_test_batches:.4f}")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Time per batch: {elapsed / smoke_test_batches:.3f} seconds")
print(
    f"Approx. training images/sec: "
    f"{(smoke_test_batches * 16) / elapsed:.2f}"
)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Model reset.
Device: cuda
Classes: ['Atelectasis', 'Effusion', 'Infiltration', 'Mass', 'Nodule']
Batch  10/100 | Loss: 1.7084
Batch  20/100 | Loss: 0.7611
Batch  30/100 | Loss: 1.4844
Batch  40/100 | Loss: 1.0929
Batch  50/100 | Loss: 1.2157
Batch  60/100 | Loss: 1.2355
Batch  70/100 | Loss: 0.9253
Batch  80/100 | Loss: 0.9411
Batch  90/100 | Loss: 0.9663
Batch 100/100 | Loss: 1.2078

SMOKE TEST COMPLETE
Average loss: 1.1647
Total time: 78.85 seconds
Time per batch: 0.788 seconds
Approx. training images/sec: 20.29
GPU memory allocated: 0.12 GB


In [33]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 2243
Validation batches: 476
Test batches: 488


In [35]:
import os

checkpoint_path = "../../models/densenet121_epoch1.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print("Checkpoint size:", round(os.path.getsize(checkpoint_path) / (1024**2), 2), "MB")

Checkpoint exists: True
Checkpoint size: 80.53 MB


In [37]:
import torch
import torch.nn as nn
from torchvision import models

# Recreate the same DenseNet121 architecture
model = models.densenet121(weights=None)

# Same 5-class output layer
model.classifier = nn.Linear(
    model.classifier.in_features,
    len(selected_classes)
)

# Move model to GPU
model = model.to(device)

# Recreate optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# Recreate loss function
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)

# Load Epoch 1 checkpoint
checkpoint_path = "../../models/densenet121_epoch1.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"]

print("Checkpoint loaded successfully!")
print("Resuming from epoch:", start_epoch)
print("Previous validation loss:", checkpoint["val_loss"])
print("Previous Macro F1:", checkpoint["macro_f1"])
print("Previous Macro ROC-AUC:", checkpoint["macro_auc"])

Checkpoint loaded successfully!
Resuming from epoch: 1
Previous validation loss: 0.9501173379165786
Previous Macro F1: 0.3423764053558643
Previous Macro ROC-AUC: 0.7944066357157183
